# Lab 8 – Quy hoạch động & Knapsack


---

## Bài 1 – Knapsack 0/1 với DP 2D + Truy vết

### 1.1. Bài toán

**Cho:**
- Danh sách `values`: giá trị từng món.
- Danh sách `weights`: khối lượng từng món.
- Sức chứa balo `W`.

**Yêu cầu:**
- Tính giá trị tối đa có thể mang.
- Xác định danh sách item được chọn (index).
- Dùng DP 2D như đã học trên lớp.

### 1.2. Hàm xây bảng dp 2D

In [ ]:
def knapsack_dp_table(values, weights, W):
    """
    Xây bảng dp 2D cho knapsack 0/1.
    dp[i][w]: max value dùng i món đầu, sức chứa w.
    """
    n = len(values)
    dp = [[0] * (W + 1) for _ in range(n + 1)]

    for i in range(1, n + 1):
        wt = weights[i - 1]
        val = values[i - 1]
        for w in range(1, W + 1):
            if wt > w:
                dp[i][w] = dp[i - 1][w]
            else:
                dp[i][w] = max(
                    dp[i - 1][w],
                    val + dp[i - 1][w - wt]
                )
    return dp

### 1.3. Hàm lấy max value từ bảng

In [ ]:
def knapsack_max_value_2d(values, weights, W):
    dp = knapsack_dp_table(values, weights, W)
    n = len(values)
    return dp[n][W]

### 1.4. Hàm truy vết danh sách item từ dp 2D

Dựa trên bảng `dp`:
- Bắt đầu từ `i = n`, `w = W`.
- Nếu `dp[i][w] == dp[i-1][w]` → item `i` không được chọn.
- Nếu khác → item `i` được chọn, trừ `w`.

In [ ]:
def knapsack_trace_items_2d(values, weights, W):
    dp = knapsack_dp_table(values, weights, W)
    n = len(values)
    w = W
    chosen = []

    i = n
    while i > 0 and w > 0:
        if dp[i][w] == dp[i - 1][w]:
            # không lấy item i-1
            i -= 1
        else:
            # lấy item i-1
            chosen.append(i - 1)
            w -= weights[i - 1]
            i -= 1

    chosen.reverse()
    max_val = dp[n][W]
    return max_val, chosen

### 1.5. Test Knapsack 2D

In [ ]:
def test_knapsack_2d():
    values  = [1, 4, 5]
    weights = [1, 2, 3]
    W = 5

    dp = knapsack_dp_table(values, weights, W)
    print("Bảng dp (hàng: i, cột: w):")
    for row in dp:
        print(row)

    max_val, items = knapsack_trace_items_2d(values, weights, W)
    print("\nMax value:", max_val)
    print("Các item chọn (index):", items)
    print("Chi tiết:")
    total_weight = sum(weights[i] for i in items)
    print("  Tổng weight:", total_weight)
    for i in items:
        print(f"  item {i}: wt={weights[i]}, val={values[i]}")

if __name__ == "__main__":
    test_knapsack_2d()

---

## Bài 2 – Knapsack 0/1 với DP 1D

### 2.1. Mục tiêu

- Viết lại knapsack 0/1 dùng mảng **1 chiều**: `dp[w]`.
- Hiểu vì sao phải **duyệt w từ W → 0** (ngược).

> **Lưu ý:** Nếu duyệt `w` từ `0 → W`, `dp[w - wt]` có thể đã được cập nhật với item `i` → dùng item `i` nhiều lần → sai với 0/1.

### 2.2. Hàm Knapsack 1D

In [ ]:
def knapsack_1d(values, weights, W):
    """
    Knapsack 0/1 dùng dp 1 chiều.
    dp[w]: max value với sức chứa w.
    """
    n = len(values)
    dp = [0] * (W + 1)

    for i in range(n):
        wt = weights[i]
        val = values[i]
        # Duyệt ngược để không dùng 1 món nhiều lần
        for w in range(W, wt - 1, -1):
            dp[w] = max(dp[w], val + dp[w - wt])

    return dp[W]

### 2.3. Test so sánh 2D vs 1D

In [ ]:
def test_knapsack_1d():
    values  = [1, 4, 5]
    weights = [1, 2, 3]
    W = 5

    ans_2d = knapsack_max_value_2d(values, weights, W)
    ans_1d = knapsack_1d(values, weights, W)

    print("Max value 2D:", ans_2d)
    print("Max value 1D:", ans_1d)

if __name__ == "__main__":
    test_knapsack_1d()

---

## Bài 3 – Knapsack 1D + Truy vết nghiệm

### 3.1. Mục tiêu

- Kết hợp lợi ích dp 1D (tối ưu value) với truy vết nghiệm.
- Cách thực hiện:
  - Dùng dp 1D cho value.
  - Dùng thêm bảng `choice[i][w]` để ghi nhận "item `i` có được chọn tại capacity `w` hay không".

### 3.2. Hàm Knapsack 1D với bảng choice

In [ ]:
def knapsack_1d_with_choice(values, weights, W):
    """
    DP 1D + bảng choice để truy vết.
    dp[w]: max value với sức chứa w.
    choice[i][w] = True nếu tại item i, capacity w, ta chọn item i.
    """
    n = len(values)
    dp = [0] * (W + 1)
    choice = [[False] * (W + 1) for _ in range(n)]

    for i in range(n):
        wt = weights[i]
        val = values[i]
        # Duyệt ngược w
        for w in range(W, wt - 1, -1):
            # Nếu lấy item i
            take_val = val + dp[w - wt]
            if take_val > dp[w]:
                dp[w] = take_val
                choice[i][w] = True
            # Nếu không lấy → dp[w] giữ nguyên, choice[i][w] = False (default)

    return dp, choice

### 3.3. Hàm truy vết items từ dp + choice

In [ ]:
def reconstruct_items_1d(values, weights, W):
    dp, choice = knapsack_1d_with_choice(values, weights, W)
    n = len(values)
    w = W
    chosen = []

    i = n - 1
    while i >= 0 and w > 0:
        if choice[i][w]:
            chosen.append(i)
            w -= weights[i]
        i -= 1

    chosen.reverse()
    max_val = dp[W]
    return max_val, chosen

### 3.4. Test truy vết nghiệm 1D

In [ ]:
def test_knapsack_1d_trace():
    values  = [1, 4, 5]
    weights = [1, 2, 3]
    W = 5

    max_val, items = reconstruct_items_1d(values, weights, W)

    print("Max value:", max_val)
    print("Các item chọn (index):", items)
    total_weight = sum(weights[i] for i in items)
    print("Tổng weight:", total_weight)
    for i in items:
        print(f"  item {i}: wt={weights[i]}, val={values[i]}")

if __name__ == "__main__":
    test_knapsack_1d_trace()